In [ ]:
# ============================================
# Ethical AI Project - Starter Notebook
# Dataset: German Credit
# Task: Predict creditworthiness
# ============================================

# -----------------------------
# 1. Install required libraries
# -----------------------------
!pip install pandas numpy scikit-learn matplotlib seaborn requests --quiet

import pandas as pd
import numpy as np
import requests
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print("Libraries installed and imported.")

# -----------------------------
# 2. Download dataset
# -----------------------------
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

data_path = data_dir / "german_credit_data.csv"
url = "https://raw.githubusercontent.com/selva86/datasets/master/GermanCredit.csv"

if not data_path.exists():
    print("Downloading German Credit dataset...")
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    with open(data_path, "wb") as f:
        f.write(r.content)
    print("Download complete.")
else:
    print("Dataset already exists.")

# -----------------------------
# 3. Load dataset
# -----------------------------
df = pd.read_csv(data_path)

print("\nDataset loaded successfully")
print("Shape:", df.shape)

print("\nFirst rows:")
print(df.head())

print("\nColumn names:")
print(df.columns)

print("\nMissing values:")
print(df.isna().sum())

# -----------------------------
# 4. Standardize column names
# -----------------------------
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

print("\nStandardized columns:")
print(df.columns)

# -----------------------------
# 5. Identify target variable robustly
# -----------------------------
possible_targets = ["credit_risk", "class", "risk", "target"]

target_col = None
for col in possible_targets:
    if col in df.columns:
        target_col = col
        break

if target_col is None:
    raise ValueError(f"Could not find target column. Tried: {possible_targets}")

print("\nDetected target column:", target_col)
print("Raw target unique values:", df[target_col].unique())

# Handle common target formats robustly
if df[target_col].dtype == "object":
    cleaned = df[target_col].astype(str).str.strip().str.lower()

    # Case 1: already encoded as strings "0"/"1"
    if set(cleaned.unique()).issubset({"0", "1"}):
        df[target_col] = cleaned.astype(int)

    # Case 2: text labels
    elif set(cleaned.unique()).issubset({"good", "bad"}):
        df[target_col] = cleaned.map({"good": 1, "bad": 0})

    else:
        raise ValueError(
            f"Unsupported target values in {target_col}: {sorted(cleaned.unique())}"
        )
else:
    # numeric target
    df[target_col] = df[target_col].astype(int)

if df[target_col].isna().any():
    raise ValueError("Target column contains NaN after conversion.")

print("\nTarget distribution:")
print(df[target_col].value_counts(dropna=False))

# -----------------------------
# 6. Create cleaner protected attributes
# -----------------------------
# Extract binary sex from personal_status_sex
if "personal_status_sex" in df.columns:
    df["sex"] = df["personal_status_sex"].apply(
        lambda x: "female" if "female" in str(x).lower() else "male"
    )

# Common fairness grouping for age
if "age" in df.columns:
    df["age_group"] = np.where(df["age"] >= 25, "age_25_and_over", "under_25")

print("\nProtected attributes available:")
for col in ["sex", "age", "age_group"]:
    if col in df.columns:
        print(f"\n{col}")
        print(df[col].value_counts())

# -----------------------------
# 7. Split dataset
# -----------------------------
X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTrain/Test split")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train distribution:")
print(y_train.value_counts())
print("y_test distribution:")
print(y_test.value_counts())

# -----------------------------
# 8. Basic preprocessing pipeline
# -----------------------------
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["int64", "float64"]).columns.tolist()

print("\nNumeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("\nProcessed training matrix shape:", X_train_processed.shape)
print("Processed test matrix shape:", X_test_processed.shape)

# Optional: keep protected attributes aside for later fairness analysis
protected_cols = [c for c in ["sex", "age", "age_group"] if c in X_train.columns]
A_train = X_train[protected_cols].copy() if protected_cols else pd.DataFrame(index=X_train.index)
A_test = X_test[protected_cols].copy() if protected_cols else pd.DataFrame(index=X_test.index)

print("\nProtected attribute preview:")
print(A_train.head())

print("\nGerman Credit dataset ready for modeling.")

In [ ]:
# ==========================================================
# HIGH-LEVEL GUIDE FOR PROJECT MODULE 1
# Do not implement everything at once.
# Work step by step after the provided starter code.
# ==========================================================

# 1. Inspect the processed dataset
#    - Check class balance in the target variable.
#    - Check the distribution of the protected attribute(s).
#    - Make sure you understand what positive and negative predictions mean.

# 2. Train a baseline model
#    - Choose a reasonable classifier (for example logistic regression,
#      decision tree, random forest, gradient boosting, etc.).
#    - Briefly justify why that model is appropriate for this task.

# 3. Evaluate predictive performance
#    - Compute accuracy, precision, recall, and F1-score on the test set.
#    - Explain what each metric tells you in the context of this dataset.

# 4. Compute fairness metrics by demographic group
#    - Compare groups separately for each protected attribute you selected.
#    - Include metrics such as:
#         * Demographic parity / selection rate differences
#         * Equalized odds components (for example TPR/FPR differences)
#         * Calibration-related comparisons if your approach supports them
#    - Clearly identify which groups have better or worse outcomes.

# 5. Visualize disparities
#    - Create plots showing:
#         * Prediction/selection rates by group
#         * Error rates by group
#         * Fairness metric gaps across groups
#    - Make plots easy to read and label them clearly.

# 6. Apply one preprocessing bias mitigation method
#    - Choose ONE:
#         * Reweighting training samples
#         * Resampling demographic groups
#    - Retrain the same baseline model after mitigation.
#    - Recompute both performance and fairness metrics.
#    - Compare the new results against the baseline.

# 7. Apply one in-processing mitigation method
#    - Choose a training-time fairness intervention such as:
#         * Fairness constraints
#         * Fairness-aware optimization
#         * Adversarial debiasing
#         * A fairness-aware ML library
#    - Retrain and evaluate again.
#    - Compare baseline vs preprocessing vs in-processing.

# 8. Analyze trade-offs
#    - Compare fairness improvements against any drop in predictive accuracy.
#    - Determine which approach gives the best balance for this dataset.
#    - Identify which fairness metric was hardest to improve.

# 9. Write deployment recommendations
#    - State whether you would recommend real-world deployment.
#    - Discuss which fairness criteria matter most in this setting.
#    - Explain possible harms if biased predictions are used in practice.
#    - Note important limitations in your analysis.

# 10. Final deliverables
#    - Completed notebook with code, outputs, and visualizations
#    - Technical report interpreting model performance, fairness results,
#      mitigation effects, trade-offs, and recommendations

In [ ]:
# Part 1: Train a baseline model
# We choose a Random Forest Classifier because it handles mixed data types well and doesn't require extensive tuning to capture non-linearities.
print("--- Part 1: Baseline Model ---")
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference

clf_baseline = RandomForestClassifier(random_state=42)
clf_baseline.fit(X_train_processed, y_train)
y_pred_baseline = clf_baseline.predict(X_test_processed)

acc = accuracy_score(y_test, y_pred_baseline)
prec = precision_score(y_test, y_pred_baseline)
rec = recall_score(y_test, y_pred_baseline)
f1 = f1_score(y_test, y_pred_baseline)

print(f"Accuracy: {acc:.3f}, Precision: {prec:.3f}, Recall: {rec:.3f}, F1: {f1:.3f}")

# Fairness Metrics
dp_diff = demographic_parity_difference(y_test, y_pred_baseline, sensitive_features=A_test)
eo_diff = equalized_odds_difference(y_test, y_pred_baseline, sensitive_features=A_test)

print(f"Demographic Parity Difference: {dp_diff:.3f}")
print(f"Equalized Odds Difference: {eo_diff:.3f}")

# Group-wise metrics
import numpy as np
for group in np.unique(A_test):
    mask = A_test.values == group
    p_rate = np.mean(y_pred_baseline[mask])
    e_rate = 1 - accuracy_score(y_test[mask], y_pred_baseline[mask])
    print(f"Group {group} -> Prediction Rate: {p_rate:.3f}, Error Rate: {e_rate:.3f}")


In [ ]:
# Part 2: Preprocessing Bias Mitigation (Resampling)
# We will upsample the groups to equal sizes per (age_group, target) combination to balance the data.
print("\n--- Part 2: Preprocessing (Resampling) ---")
import pandas as pd
X_train_dense = X_train_processed.toarray() if hasattr(X_train_processed, "toarray") else X_train_processed
train_df = pd.DataFrame(X_train_dense)
train_df['target'] = y_train.values
train_df['age_group'] = A_train.values

max_size = train_df.groupby(['age_group', 'target']).size().max()
resampled_df = pd.concat([
    group.sample(n=max_size, replace=True, random_state=42) 
    for _, group in train_df.groupby(['age_group', 'target'])
])

X_train_res = resampled_df.drop(columns=['target', 'age_group']).values
y_train_res = resampled_df['target'].values

clf_pre = RandomForestClassifier(random_state=42)
clf_pre.fit(X_train_res, y_train_res)
y_pred_pre = clf_pre.predict(X_test_processed)

acc_pre = accuracy_score(y_test, y_pred_pre)
dp_diff_pre = demographic_parity_difference(y_test, y_pred_pre, sensitive_features=A_test)
eo_diff_pre = equalized_odds_difference(y_test, y_pred_pre, sensitive_features=A_test)

print(f"Accuracy: {acc_pre:.3f}")
print(f"Demographic Parity Difference: {dp_diff_pre:.3f}")
print(f"Equalized Odds Difference: {eo_diff_pre:.3f}")


In [ ]:
# Part 3: In-processing Bias Mitigation (Exponentiated Gradient)
print("\n--- Part 3: In-processing (ExponentiatedGradient) ---")
from fairlearn.reductions import ExponentiatedGradient, DemographicParity
clf_in = RandomForestClassifier(random_state=42, max_depth=5, n_estimators=50)
constraint = DemographicParity()
mitigator = ExponentiatedGradient(clf_in, constraint, eps=0.01)
mitigator.fit(X_train_processed, y_train, sensitive_features=A_train)
y_pred_in = mitigator.predict(X_test_processed)

acc_in = accuracy_score(y_test, y_pred_in)
dp_diff_in = demographic_parity_difference(y_test, y_pred_in, sensitive_features=A_test)
eo_diff_in = equalized_odds_difference(y_test, y_pred_in, sensitive_features=A_test)

print(f"Accuracy: {acc_in:.3f}")
print(f"Demographic Parity Difference: {dp_diff_in:.3f}")
print(f"Equalized Odds Difference: {eo_diff_in:.3f}")


In [ ]:
# Part 4: Comprehensive Visualizations
import matplotlib.pyplot as plt
import seaborn as sns

groups = ["age_25_and_over", "under_25"]
models = ['Baseline', 'Preprocessing', 'Inprocessing']
clean_model_names = ['Baseline', 'Resampling', 'Exp. Gradient']

# Gather data
pred_over, pred_under = [], []
err_over, err_under = [], []

for preds in [y_pred_baseline, y_pred_pre, y_pred_in]:
    # over 25
    mask_o = A_test.values == "age_25_and_over"
    pred_over.append(np.mean(preds[mask_o]))
    err_over.append(1 - accuracy_score(y_test[mask_o], preds[mask_o]))
    
    # under 25
    mask_u = A_test.values == "under_25"
    pred_under.append(np.mean(preds[mask_u]))
    err_under.append(1 - accuracy_score(y_test[mask_u], preds[mask_u]))

x = np.arange(len(models))
width = 0.35

# 1. Prediction Rates Comparison
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width/2, pred_over, width, label='Age >= 25', color='tab:blue')
ax.bar(x + width/2, pred_under, width, label='Age < 25 (Underprivileged)', color='tab:orange')
ax.set_ylabel('Prediction/Selection Rate')
ax.set_title('Prediction Rates by Age Group Across Models')
ax.set_xticks(x)
ax.set_xticklabels(clean_model_names)
ax.legend()
plt.tight_layout()
plt.show()

# 2. Error Rates Comparison
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width/2, err_over, width, label='Age >= 25', color='tab:blue')
ax.bar(x + width/2, err_under, width, label='Age < 25 (Underprivileged)', color='tab:orange')
ax.set_ylabel('Error Rate')
ax.set_title('Error Rates by Age Group Across Models')
ax.set_xticks(x)
ax.set_xticklabels(clean_model_names)
ax.legend()
plt.tight_layout()
plt.show()

# 3. Fairness Metric Gaps Comparison (DP vs EO)
dp_gaps = [dp_diff, dp_diff_pre, dp_diff_in]
eo_gaps = [eo_diff, eo_diff_pre, eo_diff_in]

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width/2, dp_gaps, width, label='DP Difference', color='tab:green')
ax.bar(x + width/2, eo_gaps, width, label='EO Difference', color='tab:red')
ax.set_ylabel('Fairness Metric Gap (Lower is better)')
ax.set_title('Fairness Metric Gaps Across Models')
ax.set_xticks(x)
ax.set_xticklabels(clean_model_names)
ax.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# 4. Trade-off Analysis Visualization
accs = [acc, acc_pre, acc_in]

fig, ax1 = plt.subplots(figsize=(8, 5))
color = 'tab:blue'
ax1.set_xlabel('Models')
ax1.set_ylabel('Accuracy', color=color)
ax1.plot(clean_model_names, accs, marker='o', label='Accuracy', color=color, linewidth=2)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()  
color = 'tab:red'
ax2.set_ylabel('Demographic Parity Difference (Lower is better)', color=color)  
ax2.plot(clean_model_names, dp_gaps, marker='x', label='DP Difference', color=color, linewidth=2, linestyle='dashed')
ax2.tick_params(axis='y', labelcolor=color)

fig.suptitle("Trade-off between Accuracy and Fairness (Demographic Parity)")
fig.tight_layout()  
plt.show()
